In [ ]:

import xarray as xr
import plotly
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from geopy.distance import geodesic
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import re
import glob
import imageio.v3 as iio
import subprocess
from joblib import delayed, Parallel
from pace_pic_validation.viz.visualization import plot_pic_world

In [ ]:

path_06260703 = '/Users/timfugl/data/hermes_pic/ftp.hermes.acri.fr/211137319/L3m_20260626-20260703__GLOB_4_AVW-MODVIR_PIC_8D_00.nc'
path_07120719 = '/Users/timfugl/Desktop/untitled folder/L3m_20260712-20260719__GLOB_4_AVW-MODVIR_PIC_8D_00.nc'
path_07200727 = '/Users/timfugl/Desktop/untitled folder/L3m_20260720-20260727__GLOB_4_AVW-MODVIR_PIC_8D_00.nc'
path_norbjorn = '/Users/timfugl/Desktop/modist l3m PIC/merged_june_month.nc'
path_norbjornstations = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Research_Campaigns/Norbjørn /August 2026/data/Norbjorn_Stations.csv'


In [ ]:
ds_06260703 = xr.open_dataset(path_06260703)
ds_07120719 = xr.open_dataset(path_07120719)
ds_nb = xr.open_dataset(path_norbjorn)

ds_nb_stations = pd.read_csv(path_norbjornstations, sep=',', header=0, index_col=False)
ds_nb_stations = ds_nb_stations.rename(columns={"Lat": "Lon"})
ds_nb_stations = ds_nb_stations.rename(columns={"Long": "Lat"})

print(ds_nb_stations)

In [ ]:
ds_07120719 = xr.open_dataset(path_07120719)
ds_07200727 = xr.open_dataset(path_07200727)

fig = plot_pic_world(
    ds_06260703,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 80,
    lon_min = -26,
    lon_max = 20,
    title="PIC 26.6 to 3.7"
)

fig.show()

fig = plot_pic_world(
    ds_07120719,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 80,
    lon_min = -26,
    lon_max = 20,
    title="PIC 12.7 to 19.7"
)

fig.show()

fig = plot_pic_world(
    ds_07200727,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 80,
    lon_min = -26,
    lon_max = 20,
    title="PIC 20.7 to 27.7"
)

fig.show()




In [ ]:


# assign vars
sat = ds_nb
lat = sat['lat'].values        # 1D lat in degrees
lon = sat['lon'].values        # 1D lon in degrees
PIC_8D = sat['pic'].values * 1000     # 2D field [lat, lon]

lat_stat = ds_nb_stations['Lat'].values   # station lat in degrees
lon_stat = ds_nb_stations['Lon'].values   # station lon in degrees

fig, ax = plt.subplots(
    figsize=(7, 7),
    subplot_kw={'projection': ccrs.Stereographic()}
)

# base map
ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=0, alpha=0.2)
ax.add_feature(cfeature.LAND, facecolor='#90FF15', zorder=1, alpha=0.2)
ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.5, zorder=2)

# extent is given in geodetic (lon/lat) coordinates
ax.set_extent([0, 25, 65, 80], crs=ccrs.PlateCarree())

# station points (x=lon, y=lat; data CRS = PlateCarree)
ax.scatter(
    lon_stat, lat_stat,
    color='red', s=10,
    transform=ccrs.PlateCarree(),
    zorder=5, label='Norbjorn Stations'
)
ax.plot(
    lon_stat, lat_stat,
    color='red',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# station labels
for i, name in enumerate(ds_nb_stations['station']):
    ax.text(
        lon_stat[i] - 0.1, lat_stat[i] - 0.2,  # small lat offset
        name,
        fontsize=8,
        fontstyle='oblique',
        color='red',
        transform=ccrs.PlateCarree(),
        zorder=6
    )

# Create 2D coordinate grids from lon/lat
lon_2d, lat_2d = np.meshgrid(lon, lat)

# pcolormesh: data are lon/lat in degrees => PlateCarree transform
im = ax.pcolormesh(
    lon_2d, lat_2d, PIC_8D,
    transform=ccrs.PlateCarree(),
    zorder=3, vmax=2, cmap='viridis'
)

plt.colorbar(im, ax=ax, label='PIC')
ax.set_title('PIC MODIS 26.7–8.8')

plt.show()

sat = ds_07200727
lat = sat['lat'].values
lon = sat['lon'].values   
PIC_8D = sat['PIC_mean'].values * 1000


fig, ax = plt.subplots(figsize=(7, 7),
                       subplot_kw={'projection': ccrs.Stereographic()})

ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=0, alpha=0.2)
ax.add_feature(cfeature.LAND, facecolor='#90FF15', zorder=1, alpha=0.2)
ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.5, zorder=2)
ax.set_extent([0, 25, 65, 80], crs=ccrs.PlateCarree())

# station points (x=lon, y=lat; data CRS = PlateCarree)
ax.scatter(
    lon_stat, lat_stat,
    color='red', s=10,
    transform=ccrs.PlateCarree(),
    zorder=5, label='Norbjorn Stations'
)
ax.plot(
    lon_stat, lat_stat,
    color='red',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# station labels
for i, name in enumerate(ds_nb_stations['station']):
    ax.text(
        lon_stat[i] - 0.1, lat_stat[i] - 0.2,  # small lat offset
        name,
        fontsize=8,
        fontstyle='oblique',
        color='red',
        transform=ccrs.PlateCarree(),
        zorder=6
    )

    # Create 2D coordinate grids
lon_2d, lat_2d = np.meshgrid(lon, lat)

    # pcolormesh reprojects each cell correctly
im = ax.pcolormesh(lon_2d, lat_2d, PIC_8D,
                   transform=ccrs.PlateCarree(),
                   zorder=3, vmax=2, cmap='viridis')


plt.colorbar(im, ax=ax, label='PIC, mmol / m3')
ax.set_title(f'PIC 20.7-27.7')




plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

def plot_pic_world(ds, var="PIC_mean", step=8,
                   lat_min=-90, lat_max=90,
                   lon_min=-180, lon_max=180,
                   title=None):

    sub = ds.sel(
        lat=slice(lat_max, lat_min),   # descending lat
        lon=slice(lon_min, lon_max)
    ).isel(
        lat=slice(None, None, step),
        lon=slice(None, None, step)
    )

    z = sub[var].values * 1000
    lon = sub["lon"].values
    lat = sub["lat"].values

    fig = go.Figure(
        go.Heatmap(
            x=lon,
            y=lat,
            z=z,
            colorscale="Viridis",
            zmin=0,
            zmax=2,
            colorbar=dict(title=var),
            hovertemplate=(
                "lon: %{x:.3f}<br>"
                "lat: %{y:.3f}<br>"
                "PIC: %{z:.4g}<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=title or var,
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        width=1000,
        height=700,
    )

    fig.update_geos(
    resolution=50,
    showcountries=False,
    showcoastlines=True,
    coastlinecolor="black",
    showland=True,
    landcolor="lightgray"
    )


  

    return fig

In [ ]:
import xarray as xr

ds_07120719 = xr.open_dataset(path_07120719)
ds_07200727 = xr.open_dataset(path_07200727)

print(ds_07120719)
print(ds_07200727)

fig1 = plot_pic_world(
    ds_07120719,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 67,
    lon_min = -26,
    lon_max = 12,
    title="PIC 12.7 to 19.7"
)
fig1.show()

fig2 = plot_pic_world(
    ds_07200727,
    var="PIC_mean",
    step=1,
    lat_min = 52,
    lat_max = 67,
    lon_min = -26,
    lon_max = 12,
    title="PIC 20.7 to 27.7"
)
fig2.show()